# 06 — Live ingestion: one flight, its alternatives, and where its aircraft is

Two inputs. A flight number and a date. Everything else is derived.

## Why the inputs shrank

The previous version asked for a flight number, an origin, a destination and a carrier code,
and got them wrong in ways that were hard to see: one run passed `airline_iata="DL1682"`,
which returns nothing because that field wants a carrier code rather than a flight number.
Four inputs that must agree with each other is four chances to be wrong, and the failure
looks identical to "no flights today".

`AeroDataBox`'s `/flights/number/{number}/{date}` derives the itinerary from the number
alone — `DL1572` comes back as `ATL -> IAH` with scheduled and revised times, distance,
aircraft, and airline. Origin and destination are *outputs*. So they stopped being inputs.

## Why AeroDataBox and not AviationStack

The AviationStack plan in use returns schedules dated roughly two weeks in the past. That
does not merely degrade the live path, it makes it impossible: a flight from a fortnight ago
cannot be airborne now, so every attempt to match one against a live ADS-B snapshot returned
zero by construction. Every `match_rate 0.0%` in the earlier runs had that single cause.

AeroDataBox returns the flight operating on the requested date, and carries two identifiers
that matter more than the schedule itself:

| Field | Value | Why |
|---|---|---|
| `callSign` | `DAL1572` | OpenSky's callsign, supplied directly — no IATA-to-ICAO table |
| `aircraft.modeS` | `A34729` | OpenSky's `icao24` — a **unique airframe**, not a flight number |

Matching on the airframe address is the strongest join between the two feeds. Codeshares
share a flight number but not an aeroplane, and a flight number recurs daily while an
airframe address does not.

## The division of labour, unchanged

AeroDataBox answers **how late** on gate semantics — `revisedTime - scheduledTime` is the
same quantity BTS records as `DEP_DELAY` and the models were trained on. OpenSky answers
**where and what phase**. Deriving delay from ADS-B would give wheels-off, which differs
from gate delay by taxi-out and is worst at the congested airports where delay matters most.


In [ ]:
%pip install holidays -q
dbutils.library.restartPython()


In [ ]:
import sys
sys.path.append("..")

from datetime import date, datetime, time, timedelta

import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

from src import config
from src.aerodatabox import (
    AeroDataBoxClient, departure_to_row, flight_to_row, normalise_flight_number,
)
from src.opensky import (
    CONUS_BBOX, OpenSkyClient, match_by_airframe, match_to_schedule, parse_states,
    split_by_phase,
)


## The two inputs

`FLIGHT_NUMBER` is forgiving: `dl1572`, `DL 1572` and `DL-1572` all normalise to `DL1572`,
because a person typing a flight number should not have to get the spacing right.

`FLIGHT_DATE` matters more than it looks. The provider serves a window around today — a few
days back and a few days forward — so the date chooses which question is being asked. A past
date gives a flight that has already operated, with its actual times filled in. Today gives
the live picture, and it is the only choice where the OpenSky match can succeed, because
that is the only day an aircraft is where the snapshot can see it. A future date gives the
published schedule with no revisions yet, which is the pure pre-departure case.


In [ ]:
dbutils.widgets.text("FLIGHT_NUMBER", "DL1572")
dbutils.widgets.text("FLIGHT_DATE", date.today().isoformat())
dbutils.widgets.dropdown("FIND_ALTERNATIVES", "true", ["true", "false"])
dbutils.widgets.dropdown("USE_OPENSKY", "true", ["true", "false"])

FLIGHT_NUMBER = normalise_flight_number(dbutils.widgets.get("FLIGHT_NUMBER"))
FLIGHT_DATE = date.fromisoformat(dbutils.widgets.get("FLIGHT_DATE").strip())
FIND_ALTERNATIVES = dbutils.widgets.get("FIND_ALTERNATIVES") == "true"
USE_OPENSKY = dbutils.widgets.get("USE_OPENSKY") == "true"

offset = (FLIGHT_DATE - date.today()).days
horizon = ("today — the only date where a live aircraft match is possible"
           if offset == 0 else
           f"{abs(offset)} day(s) {'ahead' if offset > 0 else 'behind'}")

print(f"Flight : {FLIGHT_NUMBER}")
print(f"Date   : {FLIGHT_DATE}  ({horizon})")
if offset != 0:
    print("\n  The OpenSky snapshot is always live. Matching a flight from another")
    print("  day against it cannot succeed, so the phase split will be 'unknown'")
    print("  and everything routes to the pre-departure model. That is correct")
    print("  behaviour, not a failure — set the date to today to exercise the join.")


## The flight of interest

In [ ]:
adb = AeroDataBoxClient(
    dbutils.secrets.get(config.AERODATABOX_SECRET_SCOPE, config.AERODATABOX_SECRET_KEY)
)

legs = adb.flight_by_number(FLIGHT_NUMBER, FLIGHT_DATE)
print(f"AeroDataBox returned {len(legs)} leg(s) for {FLIGHT_NUMBER} on {FLIGHT_DATE}")
print(f"Quota: {adb.last_quota.get('units_remaining')}/{adb.last_quota.get('units_limit')} "
      f"units, {adb.last_quota.get('requests_remaining')}/"
      f"{adb.last_quota.get('requests_limit')} requests remaining")

focus_rows = [r for r in (flight_to_row(leg) for leg in legs) if r]
if not focus_rows:
    raise ValueError(
        f"{FLIGHT_NUMBER} has no usable record on {FLIGHT_DATE}. The provider serves a "
        "window around today; try a date within a few days, or check the number."
    )

for r in focus_rows:
    delay = "not yet departed" if r["dep_delay"] is None else f"{r['dep_delay']:+.0f} min"
    print(f"\n  {r['flight_iata']}  {r['origin_airport_code']} -> "
          f"{r['destination_airport_code']}  on {r['flight_date']}")
    print(f"    scheduled departure {r['crs_dep_time']:04d}Z   "
          f"arrival {r['crs_arr_time']:04d}Z   {r['distance']:.0f} km")
    print(f"    gate departure delay: {delay}")
    print(f"    aircraft {r['aircraft_reg']} ({r['aircraft_model']})  "
          f"callsign {r['flight_icao']}  icao24 {r['aircraft_icao24']}")
    print(f"    status {r['flight_status']}   quality {r['data_quality']}")

focus = focus_rows[0]
ORIGIN, DESTINATION = focus["origin_airport_code"], focus["destination_airport_code"]
print(f"\nRoute derived from the flight number: {ORIGIN} -> {DESTINATION}")
print("The user never typed either of those.")


## Alternatives on the same route

One call to the origin airport, windowed around the flight's own departure, filtered to the
same destination. Codeshares are excluded at the source: one aircraft sold under three flight
numbers would otherwise appear as three alternatives to itself, which is what the earlier
AviationStack runs produced — `DL753`, `WS6993` and `AM4626` were one aeroplane leaving ATL
at 18:47.


In [ ]:
alt_rows = []
if not FIND_ALTERNATIVES:
    print("FIND_ALTERNATIVES=false — scoring the flight of interest alone.")
else:
    dep_local = datetime.combine(
        focus["flight_date"], time(focus["crs_dep_time"] // 100, focus["crs_dep_time"] % 100)
    )
    span = timedelta(hours=config.ALTERNATIVE_SEARCH_HOURS)
    window_start, window_end = dep_local - span, dep_local + span

    departures = adb.airport_departures(ORIGIN, window_start, window_end)
    print(f"{ORIGIN} departures in +/-{config.ALTERNATIVE_SEARCH_HOURS}h: {len(departures)}")
    print(f"Quota: {adb.last_quota.get('units_remaining')} units remaining")

    candidates = [r for r in (departure_to_row(d, ORIGIN) for d in departures) if r]
    same_route = [r for r in candidates if r["destination_airport_code"] == DESTINATION]
    print(f"  of which {ORIGIN} -> {DESTINATION}: {len(same_route)}")

    focus_numbers = {r["flight_iata"] for r in focus_rows}
    alt_rows = [r for r in same_route if r["flight_iata"] not in focus_numbers]
    print(f"  excluding the flight of interest itself: {len(alt_rows)} alternatives")

    if not alt_rows and same_route:
        print("\n  Every same-route departure in the window IS the flight of interest.")
    elif not alt_rows:
        print(f"\n  Nothing else flies {ORIGIN} -> {DESTINATION} in this window. Widen")
        print("  ALTERNATIVE_SEARCH_HOURS in config, or accept that the route is thin.")


## Assemble the pool

In [ ]:
rows = [dict(r, is_flight_of_interest=True) for r in focus_rows]
rows += [dict(r, is_flight_of_interest=False) for r in alt_rows]

silver_pdf = pd.DataFrame(rows)

# Same route, same minute, two operating carriers is a codeshare the provider's
# filter missed rather than a genuine choice.
dupe_key = ["origin_airport_code", "destination_airport_code", "flight_date", "crs_dep_time"]
dupes = silver_pdf.duplicated(subset=dupe_key, keep="first").sum()
if dupes:
    silver_pdf = (
        silver_pdf.sort_values("is_flight_of_interest", ascending=False)
        .drop_duplicates(subset=dupe_key, keep="first")
        .copy()
    )
    print(f"Collapsed {dupes} same-minute duplicate(s) on the same route")

HAS_ROWS = len(silver_pdf) > 0
print(f"Pool: {len(silver_pdf)} flights "
      f"({int(silver_pdf['is_flight_of_interest'].sum())} of interest, "
      f"{int((~silver_pdf['is_flight_of_interest']).sum())} alternatives)")

known = silver_pdf["dep_delay"].notna().sum()
print(f"Carrying a gate departure delay: {known} of {len(silver_pdf)}")
print("  Those can use the in-flight model. The rest get pre-departure, which is")
print("  the only variant that works before an aircraft has left.")


## Raw payload to Bronze

In [ ]:
import json as _json

raw = _json.dumps({"flight": legs, "alternatives_count": len(alt_rows),
                   "flight_number": FLIGHT_NUMBER, "flight_date": FLIGHT_DATE.isoformat()})
(
    spark.createDataFrame([(raw,)], ["raw_payload"])
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source", lit("aerodatabox"))
    .write.format("delta").mode("append").option("mergeSchema", "true")
    .saveAsTable(config.API_BRONZE)
)
print(f"Appended raw payload -> {config.API_BRONZE}")


## Live aircraft state

One call returns the whole tracked airspace. The match is on `icao24`, the airframe address
AeroDataBox gave us as `aircraft.modeS` — a specific aeroplane rather than a flight number.
Callsign matching runs as a fallback for flights with no tail assigned yet.


In [ ]:
state_rows, phase_by_key, opensky_report = [], {}, None

if not USE_OPENSKY:
    print("USE_OPENSKY=false — skipping the live state feed.")
else:
    try:
        opensky = OpenSkyClient(
            dbutils.secrets.get(config.OPENSKY_SECRET_SCOPE, config.OPENSKY_CLIENT_ID_KEY),
            dbutils.secrets.get(config.OPENSKY_SECRET_SCOPE, config.OPENSKY_CLIENT_SECRET_KEY),
        )
        state_rows = parse_states(opensky.fetch_states(CONUS_BBOX))
        ground, airborne = split_by_phase(state_rows)
        print(f"OpenSky: {len(state_rows):,} aircraft in ONE call "
              f"(token refreshes: {opensky.refresh_count})")
        print(f"  on ground {len(ground):,}   airborne {len(airborne):,}")

        airframes = [a for a in silver_pdf.get("aircraft_icao24", []) if a]
        by_frame, frame_report = match_by_airframe(state_rows, airframes)
        print(f"\nAirframe match (icao24): {frame_report['matched']} of "
              f"{frame_report['airframes_wanted']} aircraft located")

        callsigns = [c for c in silver_pdf.get("flight_icao", []) if c]
        by_call, call_report = match_to_schedule(state_rows, callsigns)
        print(f"Callsign match (fallback): {call_report['matched']} of "
              f"{call_report['scheduled_flights']} flights located")

        frame_to_phase = {
            (r.get("icao24") or "").lower():
                ("airborne" if r["on_ground"] is not True else "on_ground")
            for r in by_frame
        }
        call_to_phase = {
            r["callsign"]: ("airborne" if r["on_ground"] is not True else "on_ground")
            for r in by_call
        }
        phase_by_key = {"airframe": frame_to_phase, "callsign": call_to_phase}
        opensky_report = {"airframe": frame_report, "callsign": call_report}

        if not frame_to_phase and not call_to_phase:
            print("\n  Nothing matched. Expected unless the date is today and the")
            print("  aircraft is moving: a parked aeroplane with its transponder off")
            print("  broadcasts nothing, and a flight on another date is not in a live")
            print("  snapshot at all.")
    except Exception as e:
        print(f"OpenSky unavailable ({type(e).__name__}: {e}).")
        print("Continuing — every flight falls back to pre-departure.")


## Attach phase, then write

In [ ]:
if state_rows:
    (
        spark.createDataFrame(pd.DataFrame(state_rows))
        .withColumn("ingested_at", current_timestamp())
        .write.format("delta").mode("append").option("mergeSchema", "true")
        .saveAsTable(config.OPENSKY_STATES)
    )
    print(f"Appended {len(state_rows):,} state vectors -> {config.OPENSKY_STATES}")

# Airframe first, callsign second, 'unknown' last. Written only after the phase is
# attached: an earlier version wrote the table two cells before this and the column
# never reached it.
frame_map = (phase_by_key or {}).get("airframe", {})
call_map = (phase_by_key or {}).get("callsign", {})


def _phase(row):
    frame = (row.get("aircraft_icao24") or "").lower()
    if frame and frame in frame_map:
        return frame_map[frame]
    call = row.get("flight_icao")
    if call and call in call_map:
        return call_map[call]
    return "unknown"


silver_pdf["flight_phase"] = [_phase(r) for _, r in silver_pdf.iterrows()]
print(f"Phase: {silver_pdf['flight_phase'].value_counts().to_dict()}")

silver_sdf = (
    spark.createDataFrame(silver_pdf)
    .withColumn("ingested_at", current_timestamp())
)
(
    silver_sdf.write.format("delta").mode("append")
    .option("mergeSchema", "true").saveAsTable(config.API_SILVER)
)
print(f"Appended {silver_sdf.count()} rows to {config.API_SILVER} (with flight_phase)")
display(silver_sdf.select(
    "is_flight_of_interest", "flight_iata", "origin_airport_code",
    "destination_airport_code", "crs_dep_time", "dep_delay", "flight_phase",
    "aircraft_icao24", "flight_status",
).orderBy("crs_dep_time"))
